# Technological Diffusion Analysis
## Capstone Project Notebook

The adoption of new technologies — from smartphones to electric vehicles, from the internet to social media — consistently follows a characteristic **S-shaped curve**. Initially, only a few innovators adopt. Then, as awareness spreads through social networks and the technology proves its value, adoption accelerates rapidly. Finally, the market saturates as most potential adopters have already switched.

This pattern was first systematically studied by Everett Rogers in his landmark 1962 book *Diffusion of Innovations*, which identified five adopter categories: **innovators** (2.5%), **early adopters** (13.5%), **early majority** (34%), **late majority** (34%), and **laggards** (16%). The cumulative adoption curve over these groups naturally forms an S-shape.

### Why Model Technology Diffusion?

Understanding and predicting technology adoption has enormous practical value:

- **Business strategy** — forecasting market size, timing product launches, planning production capacity
- **Policy making** — predicting the adoption of renewable energy, electric vehicles, or digital infrastructure to guide investment and regulation
- **Historical analysis** — understanding why some technologies spread rapidly while others stall
- **Sustainability transitions** — modelling how quickly societies can shift to cleaner technologies

### Mathematical Models

Two classical models capture the S-curve:

1. **Logistic growth model** — a simple three-parameter curve ($M$, $k$, $t_0$) that fits many adoption datasets well but treats adoption as a homogeneous process.

2. **Bass diffusion model** (1969) — distinguishes between **innovators** (who adopt independently, driven by external influences like advertising) and **imitators** (who adopt due to word-of-mouth from existing adopters). This distinction gives the Bass model a more mechanistic foundation and often produces better fits for the early adoption phase.

### Data Sources

Finding good adoption data is a challenge in itself. Useful sources include:
- [Our World in Data](https://ourworldindata.org/) — technology adoption, internet, energy
- [ITU Statistics](https://www.itu.int/en/ITU-D/Statistics/) — telecom data
- [World Bank Open Data](https://data.worldbank.org/) — various indicators
- National statistical offices, industry reports, academic papers

### In this notebook you will:
1. Understand how technology adoption follows **S-shaped curves**
2. Implement the **logistic growth model** for diffusion
3. Implement the **Bass diffusion model** as an alternative
4. Fit both models to synthetic data using `scipy.optimize.curve_fit`
5. Learn goodness-of-fit metrics ($R^2$, RMSE, MAPE)
6. Lay the groundwork for fitting real-world technology adoption data

---

## 0 · Setup

We import NumPy for numerical computation, Matplotlib for plotting, and `curve_fit` from SciPy for nonlinear least-squares fitting. Model fitting is a central skill in this project — you will use it to estimate parameters from real-world adoption data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 12})
print('Imports loaded.')

---
## 1 · The S-Curve of Technology Adoption

Technology adoption typically follows three phases:
1. **Early adoption** — slow growth, innovators and early adopters experiment with the new technology
2. **Rapid growth** — mainstream adoption accelerates through network effects and word-of-mouth
3. **Saturation** — the market is nearly fully penetrated, growth slows asymptotically

This S-shaped pattern is mathematically captured by the **logistic function**:

$$N(t) = \frac{M}{1 + \exp\left(-k(t - t_0)\right)}$$

| Parameter | Meaning |
|---|---|
| $M$ | Market potential (carrying capacity) — the total number of potential adopters |
| $k$ | Growth rate (steepness of the curve) — how rapidly adoption accelerates |
| $t_0$ | Inflection point — the time of fastest growth (where the curve switches from convex to concave) |

The plot below shows how the growth rate $k$ affects the curve shape. A larger $k$ means faster transition from early adoption to saturation — the technology "takes off" more explosively. The inflection point $t_0$ marks the moment when half the market has adopted and the growth rate begins to slow.

In [ ]:
def logistic_model(t, M, k, t0):
    """Logistic growth model for cumulative adoption."""
    return M / (1 + np.exp(-k * (t - t0)))

# Visualise
t_vis = np.linspace(0, 30, 300)
fig, ax = plt.subplots(figsize=(10, 5))
for k_val, col in zip([0.3, 0.5, 1.0], ['#3498db', '#e67e22', '#e74c3c']):
    ax.plot(t_vis, logistic_model(t_vis, M=100, k=k_val, t0=15),
            lw=2, color=col, label=f'k = {k_val}')
ax.axhline(100, color='gray', ls=':', lw=1)
ax.axvline(15, color='gray', ls=':', lw=1, label='Inflection $t_0 = 15$')
ax.set_xlabel('Time (years)'); ax.set_ylabel('Cumulative adopters')
ax.set_title('Logistic S-Curves with Different Growth Rates', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

---
## 2 · The Bass Diffusion Model

The Bass model (1969) adds a crucial distinction that the logistic model lacks: it separates adopters into **innovators** (who adopt based on external influences like advertising or personal curiosity) and **imitators** (who adopt because they see others using the technology — word-of-mouth, social proof).

$$\frac{dN}{dt} = \left(p + q\frac{N(t)}{M}\right)\left(M - N(t)\right)$$

| Parameter | Meaning | Typical range |
|---|---|---|
| $p$ | Innovation coefficient (external influence) | 0.01–0.03 |
| $q$ | Imitation coefficient (internal influence) | 0.3–0.5 |
| $M$ | Market potential | dataset-dependent |

When $N$ is small, the term $p(M - N)$ dominates — early adoption is driven by innovators. As $N$ grows, the imitation term $q(N/M)(M - N)$ takes over, creating the rapid acceleration phase. The ratio $q/p$ determines the "peakedness" of the adoption rate curve.

The closed-form solution is:

$$N(t) = M \cdot \frac{1 - e^{-(p+q)t}}{1 + \frac{q}{p}e^{-(p+q)t}}$$

Below we compare both models visually. Notice how the Bass model has a more gradual initial take-off (driven by the small $p$) compared to the logistic model.

In [ ]:
def bass_model(t, M, p, q):
    """Bass diffusion model — cumulative adoption."""
    return M * (1 - np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))

# Compare logistic vs Bass
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_vis, logistic_model(t_vis, 100, 0.5, 15), 'b-', lw=2, label='Logistic')
ax.plot(t_vis, bass_model(t_vis, 100, 0.02, 0.4), 'r--', lw=2, label='Bass (p=0.02, q=0.4)')
ax.set_xlabel('Time'); ax.set_ylabel('Cumulative adopters')
ax.set_title('Logistic vs Bass Diffusion Model', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()
print('The Bass model captures the initial slow uptake by innovators more realistically.')

---
## 3 · Fitting Models to Data

The real power of these models lies in fitting them to **observed data**. Given a time series of cumulative adoption numbers, we use `scipy.optimize.curve_fit` to find the parameter values ($M$, $k$, $t_0$ for logistic; $M$, $p$, $q$ for Bass) that minimise the sum of squared residuals.

Below we create a synthetic dataset resembling smartphone adoption (in millions of users) to demonstrate the fitting workflow. In your project, you will replace this with real data from sources like Our World in Data or the World Bank.

In [ ]:
# Synthetic "smartphone adoption" data (millions of users)
t_data = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], dtype=float)
N_data = np.array([2, 5, 12, 28, 55, 95, 150, 220, 310, 400, 480, 540, 580, 600, 610, 615], dtype=float)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(t_data, N_data, s=60, color='black', zorder=3, label='Data')
ax.set_xlabel('Year'); ax.set_ylabel('Users (millions)')
ax.set_title('Synthetic Technology Adoption Data', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

### Fit the logistic model

We use `curve_fit` with initial guesses for the three parameters. Good initial guesses are important for convergence — we estimate $M$ from the maximum observed value, $k \approx 0.5$, and $t_0$ as roughly the midpoint of the time range. The Bass model requires initial guesses for $p$ (small, around 0.02) and $q$ (larger, around 0.4), reflecting the typical finding that imitation drives most adoption.

In [ ]:
popt_log, pcov_log = curve_fit(logistic_model, t_data, N_data,
                                  p0=[600, 0.5, 7], maxfev=5000)
M_log, k_log, t0_log = popt_log
print(f'Logistic fit: M={M_log:.1f}, k={k_log:.3f}, t0={t0_log:.1f}')

popt_bass, pcov_bass = curve_fit(bass_model, t_data, N_data,
                                  p0=[600, 0.02, 0.4], maxfev=5000)
M_bass, p_bass, q_bass = popt_bass
print(f'Bass fit: M={M_bass:.1f}, p={p_bass:.4f}, q={q_bass:.3f}')

### Goodness of fit

To compare models quantitatively, we compute three standard metrics:

- **$R^2$ (coefficient of determination)** — fraction of variance explained by the model. Values close to 1.0 indicate an excellent fit. Computed as $R^2 = 1 - SS_\text{res}/SS_\text{tot}$.
- **RMSE (root mean square error)** — average magnitude of prediction error, in the same units as the data. Lower is better.
- **MAPE (mean absolute percentage error)** — average percentage deviation from observed values. Useful for comparing across datasets with different scales, but sensitive to small observed values near zero.

In [ ]:
def goodness_of_fit(observed, predicted):
    ss_res = np.sum((observed - predicted) ** 2)
    ss_tot = np.sum((observed - np.mean(observed)) ** 2)
    r2 = 1 - ss_res / ss_tot
    rmse = np.sqrt(np.mean((observed - predicted) ** 2))
    mape = np.mean(np.abs((observed - predicted) / observed)) * 100
    return r2, rmse, mape

N_pred_log = logistic_model(t_data, *popt_log)
N_pred_bass = bass_model(t_data, *popt_bass)

r2_l, rmse_l, mape_l = goodness_of_fit(N_data, N_pred_log)
r2_b, rmse_b, mape_b = goodness_of_fit(N_data, N_pred_bass)

print(f'{"":12s} {"R²":>8s} {"RMSE":>8s} {"MAPE%":>8s}')
print(f'{"Logistic":12s} {r2_l:>8.4f} {rmse_l:>8.2f} {mape_l:>8.2f}')
print(f'{"Bass":12s} {r2_b:>8.4f} {rmse_b:>8.2f} {mape_b:>8.2f}')

### Visual comparison

The left panel overlays both fitted curves on the data points. Both models typically fit S-shaped data well, but differences appear in the tails — the Bass model may better capture the initial slow uptake, while the logistic model may fit the saturation phase more tightly.

The right panel shows **residuals** (observed minus predicted) for both models. Systematic patterns in the residuals (e.g., all positive then all negative) indicate model misspecification — the model is missing some feature of the data. Random, evenly distributed residuals indicate a good fit.

In [ ]:
t_smooth = np.linspace(0, 20, 300)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(t_data, N_data, s=50, color='black', zorder=3)
ax1.plot(t_smooth, logistic_model(t_smooth, *popt_log), 'b-', lw=2, label=f'Logistic ($R^2$={r2_l:.4f})')
ax1.plot(t_smooth, bass_model(t_smooth, *popt_bass), 'r--', lw=2, label=f'Bass ($R^2$={r2_b:.4f})')
ax1.set_xlabel('Year'); ax1.set_ylabel('Users (M)')
ax1.set_title('Model Fits', fontweight='bold'); ax1.legend(fontsize=9)

# Residuals
ax2.bar(t_data - 0.15, N_data - N_pred_log, width=0.3, color='blue', alpha=0.6, label='Logistic')
ax2.bar(t_data + 0.15, N_data - N_pred_bass, width=0.3, color='red', alpha=0.6, label='Bass')
ax2.axhline(0, color='gray', lw=1)
ax2.set_xlabel('Year'); ax2.set_ylabel('Residual')
ax2.set_title('Residual Plot', fontweight='bold'); ax2.legend(fontsize=9)

plt.tight_layout(); plt.show()

---
## 4 · Your Tasks

### Required
1. **Data Collection**: Find at least **two real-world datasets** tracking technology adoption over time (e.g., mobile phones, internet, electric cars, social media). Start early — finding good data is a challenge.
2. **Logistic Model Fit**: Fit the logistic model, report parameters and goodness of fit.
3. **Alternative Model**: Fit the Bass diffusion model (or another) and compare.

### Discussion points
- Interpret $M$, $k$, $t_0$ in terms of the technology's real-world adoption story
- Compare logistic vs Bass — which fits better and why?
- Discuss limitations (e.g., external shocks, competing technologies)
- Project forward: what does the model predict for the next 10 years? Is it plausible?

### Data sources
- [Our World in Data](https://ourworldindata.org/) — technology adoption, internet, energy
- [ITU Statistics](https://www.itu.int/en/ITU-D/Statistics/) — telecom data
- [World Bank Open Data](https://data.worldbank.org/) — various indicators

In [ ]:
# ============================================================
# PLACEHOLDER: Implement your analysis with real data below
# ============================================================
# Load real data   — TODO
# Fit logistic     — TODO
# Fit Bass         — TODO
# Compare & report — TODO

---
## Recommended Reading & Journal Club

**1. Bass, F. M. (1969)** *A new product growth for model consumer durables.* Management Science, 15(5), 215–227. [DOI](https://doi.org/10.1287/mnsc.15.5.215)
→ The original Bass diffusion model paper — one of the most cited in marketing.

**2. Rogers, E. M. (2003)** *Diffusion of Innovations.* 5th ed. Free Press.
→ The classic textbook on innovation diffusion theory.

**3. Meade, N. & Islam, T. (2006)** *Modelling and forecasting the diffusion of innovation — A 25-year review.* International Journal of Forecasting, 22(3), 519–545. [DOI](https://doi.org/10.1016/j.ijforecast.2006.01.005)
→ Comprehensive review of diffusion models including extensions of Bass.

**4. Grubler, A. (1991)** *Diffusion: Long-term patterns and discontinuities.* Technological Forecasting and Social Change, 39(1–2), 159–180.
→ Long-term historical analysis of technology diffusion patterns including railways and energy systems.